In [1]:
!pip install jsonlines

In [2]:
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class THUMOS14_I3D_Dataset(Dataset):
    def __init__(self, data_root, split='train', fps=25.0, clip_length=16):
        self.data_root = data_root
        self.split = split
        self.fps = fps
        self.clip_length = clip_length
        
        self.rgb_dir = os.path.join(data_root, 'features', split, 'rgb')
        self.flow_dir = os.path.join(data_root, 'features', split, 'flow')
        
        split_file = os.path.join(data_root, f'split_{split}.txt')
        with open(split_file, 'r') as f:
            self.video_ids = [line.strip() for line in f.readlines()]
            
        with open(os.path.join(data_root, 'gt.json'), 'r') as f:
            self.ground_truth = json.load(f)

        # --- THE RAM CACHE ---
        self.data_cache = []
        print(f"Loading {split} dataset into RAM. This will take about 60 seconds...")
        
        for vid_id in tqdm(self.video_ids, desc=f"Caching {split} Features"):
            try:
                rgb_feat = np.load(os.path.join(self.rgb_dir, f'{vid_id}.npy'))
                flow_feat = np.load(os.path.join(self.flow_dir, f'{vid_id}.npy'))
                fused_features = torch.tensor(np.concatenate((rgb_feat, flow_feat), axis=-1), dtype=torch.float32)
            except FileNotFoundError:
                continue # Skip missing videos silently during caching
                
            num_vectors = fused_features.shape[0] 
            target_mask = torch.zeros(num_vectors, dtype=torch.float32)
            
            if vid_id in self.ground_truth['database']:
                annotations = self.ground_truth['database'][vid_id]['annotations']
                for ann in annotations:
                    start_sec = float(ann['segment'][0])
                    end_sec = float(ann['segment'][1])
                    
                    start_idx = int((start_sec * self.fps) / self.clip_length)
                    end_idx = int((end_sec * self.fps) / self.clip_length)
                    
                    start_idx = max(0, min(start_idx, num_vectors - 1))
                    end_idx = max(0, min(end_idx, num_vectors - 1))
                    
                    if start_idx <= end_idx:
                        target_mask[start_idx:end_idx + 1] = 1.0
            
            # Store the fully processed tensors in memory
            self.data_cache.append((fused_features, target_mask, vid_id))

    def __len__(self):
        return len(self.data_cache)

    def __getitem__(self, idx):
        # Instantly return the pre-loaded tensors from RAM
        return self.data_cache[idx]

In [3]:
import os 
os.cpu_count()

4

In [4]:
# 1. Update the DATA_ROOT to include the nested THUMOS14 folder
DATA_ROOT = '/kaggle/input/datasets/valuejack/thumos14-asl/THUMOS14' 

# 2. Run the Dataloader again
train_dataset = THUMOS14_I3D_Dataset(DATA_ROOT, split='train')

train_loader = DataLoader(
    train_dataset, 
    batch_size=1, 
    shuffle=True,
    num_workers=0, # Reset to 0
    pin_memory=True 
)

# 3. Fetch just one video to verify
features, mask, vid_id = next(iter(train_loader))

print(f"Video ID: {vid_id[0]}")
print(f"Feature Tensor Shape: {features.shape} -> (Batch, Sequence_Length, Features)")
print(f"Mask Tensor Shape: {mask.shape} -> (Batch, Sequence_Length)")
print(f"Total Action Vectors in this video: {int(mask.sum().item())} out of {mask.shape[1]}")

Loading train dataset into RAM. This will take about 60 seconds...


Caching train Features:   0%|          | 0/200 [00:00<?, ?it/s]

Video ID: video_validation_0000788
Feature Tensor Shape: torch.Size([1, 105, 2048]) -> (Batch, Sequence_Length, Features)
Mask Tensor Shape: torch.Size([1, 105]) -> (Batch, Sequence_Length)
Total Action Vectors in this video: 41 out of 105


In [5]:
next(iter(train_loader))

[tensor([[[0.1657, 0.0458, 0.1203,  ..., 0.0111, 0.0265, 0.4205],
          [0.1874, 0.0830, 0.1322,  ..., 0.1097, 0.0164, 0.0000],
          [0.2767, 0.0671, 0.1346,  ..., 0.3108, 0.0000, 0.1381],
          ...,
          [0.1039, 0.0683, 0.1077,  ..., 0.1088, 0.0510, 0.0000],
          [0.1131, 0.0801, 0.1641,  ..., 0.0000, 0.0873, 0.0000],
          [0.1131, 0.0801, 0.1641,  ..., 0.0000, 0.0873, 0.0000]]]),
 tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
          1., 1., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
          1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.,
          0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
          1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
          1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
          1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
          1.

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        """
        alpha: Balances the importance of positive/negative examples.
               Since actions are rare, alpha gives them more weight.
        gamma: The focusing parameter. Higher values penalize the model 
               more for missing the rare action frames.
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs are raw logits before sigmoid
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss) # Probability of the correct class
        
        # Calculate Focal Loss
        F_loss = self.alpha * (1 - pt)**self.gamma * BCE_loss

        if self.reduction == 'mean':
            return torch.mean(F_loss)
        elif self.reduction == 'sum':
            return torch.sum(F_loss)
        else:
            return F_loss

In [7]:
import torch
import torch.nn as nn
from transformers import MambaConfig, MambaModel

class BiMambaGlobalScanner(nn.Module):
    def __init__(self, input_dim=2048, d_model=256, d_state=16, d_conv=4, expand=2, kernel_size=7):
        super(BiMambaGlobalScanner, self).__init__()
        
        # 1. Dimensionality Reduction
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # 2. Temporal Smoothing Bottleneck (1D Depthwise Convolution)
        # We use padding=kernel_size//2 to ensure the sequence length doesn't change
        self.temporal_smooth = nn.Conv1d(
            in_channels=d_model, 
            out_channels=d_model, 
            kernel_size=kernel_size, 
            padding=kernel_size // 2,
            groups=d_model # Depthwise convolution keeps it extremely lightweight
        )
        self.norm = nn.LayerNorm(d_model)
        
        # 3. The Core Mamba Blocks (Bidirectional)
        config = MambaConfig(
            hidden_size=d_model,
            state_size=d_state,
            conv_kernel=d_conv,
            expand=expand
        )
        self.mamba_fwd = MambaModel(config)
        self.mamba_bwd = MambaModel(config)
        
        # 4. The Classification Head
        # Note: d_model is multiplied by 2 because we concatenate forward and backward states
        self.classifier = nn.Linear(d_model * 2, 1)

    def forward(self, x):
        # x shape: (Batch, Sequence_Length, 2048)
        
        # Project inputs
        x = self.input_proj(x)  # Shape: (Batch, Sequence_Length, 256)
        
        # Apply Temporal Smoothing
        # Conv1d expects shape (Batch, Channels, Sequence_Length), so we transpose
        x_t = x.transpose(1, 2)
        x_t = self.temporal_smooth(x_t)
        x_smoothed = x_t.transpose(1, 2)
        
        # Add a residual connection and normalize to stabilize training
        x = self.norm(x + x_smoothed)
        
        # Forward Temporal Scan
        out_fwd = self.mamba_fwd(inputs_embeds=x).last_hidden_state
        
        # Backward Temporal Scan
        # Flip the sequence along the time dimension (dim=1)
        x_reversed = torch.flip(x, dims=[1])
        out_bwd = self.mamba_bwd(inputs_embeds=x_reversed).last_hidden_state
        
        # Flip the backward output back to the original chronological order
        out_bwd = torch.flip(out_bwd, dims=[1])
        
        # Combine past and future context! 
        hidden_states = torch.cat([out_fwd, out_bwd], dim=-1) # Shape: (Batch, Sequence, 512)
        
        # Classify each timestep
        logits = self.classifier(hidden_states) # Shape: (Batch, Sequence, 1)
        
        # Remove the last dimension so it matches the target mask shape
        return logits.squeeze(-1)

In [8]:
# Instantiate the UPGRADED model
model = BiMambaGlobalScanner()

# Run our single batch from the Dataloader Sanity Check through the model
predicted_logits = model(features)

print(f"Input Feature Shape: {features.shape}")
print(f"Output Logits Shape: {predicted_logits.shape}")
print(f"Target Mask Shape:   {mask.shape}")
print("\nSuccess! The shapes match perfectly.")

The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.


Input Feature Shape: torch.Size([1, 105, 2048])
Output Logits Shape: torch.Size([1, 105])
Target Mask Shape:   torch.Size([1, 105])

Success! The shapes match perfectly.


In [ ]:
import torch
import torch.optim as optim
from tqdm.auto import tqdm
import torch.amp

# Hyperparameters
EPOCHS = 10
LEARNING_RATE = 1e-4
MAX_SEQ_LEN = 1024  # NEW: Strict memory cap (approx 40 seconds of video)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Training on: {DEVICE}")

# Initialize Model, Loss, and Optimizer
mamba_model = BiMambaGlobalScanner().to(DEVICE)
criterion = FocalLoss(alpha=0.85, gamma=2.0).to(DEVICE) 
optimizer = optim.AdamW(mamba_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Initialize the Gradient Scaler for Mixed Precision
scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_recall = 0
    valid_batches = 0 

    pbar = tqdm(dataloader, desc="Training Epoch")

    for features, targets, _ in pbar:
        # --- NEW: Dynamic Sequence Chunking ---
        seq_len = features.shape[1]
        if seq_len > MAX_SEQ_LEN:
            # Pick a random starting frame that leaves enough room for the max length
            start_idx = torch.randint(0, seq_len - MAX_SEQ_LEN, (1,)).item()
            # Crop both the features and the targets
            features = features[:, start_idx : start_idx + MAX_SEQ_LEN, :]
            targets = targets[:, start_idx : start_idx + MAX_SEQ_LEN]
        # --------------------------------------

        features, targets = features.to(device), targets.to(device)
        optimizer.zero_grad()

        # Cast operations to mixed precision
        with torch.amp.autocast('cuda'):
            logits = model(features)
            loss = criterion(logits, targets)
        
        # Scale the loss and call backward
        scaler.scale(loss).backward()
        
        # Step the optimizer and update the scaler
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float() 

            actual_positives = targets.sum().item()
            if actual_positives > 0:
                true_positives = (preds * targets).sum().item()
                recall = true_positives / actual_positives
                total_recall += recall
                valid_batches += 1
            else:
                recall = 0.0 

        pbar.set_postfix(Loss=f"{loss.item():.4f}", Recall=f"{recall:.4f}")

    avg_loss = total_loss / len(dataloader)
    avg_recall = total_recall / valid_batches if valid_batches > 0 else 0
    
    return avg_loss, avg_recall

for epoch in range(1, EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{EPOCHS} ---")
    avg_loss, avg_recall = train_epoch(mamba_model, train_loader, optimizer, criterion, DEVICE)
    print(f"Epoch {epoch} Complete | Avg Loss: {avg_loss:.4f} | Avg Recall: {avg_recall:.4f}")

torch.save(mamba_model.state_dict(), "mamba_scanner_stage1.pth")
print("\nModel saved successfully!")

Training on: cuda

--- Epoch 1/10 ---


Training Epoch:   0%|          | 0/200 [00:00<?, ?it/s]

In [ ]:
# Assuming DATA_ROOT is still pointing to your THUMOS14 folder
refiner_dataset = TransformerRefinerDataset(DATA_ROOT, split='train', window_size=64)

# We can use a larger batch size now because the sequences are only 64 frames long!
refiner_loader = DataLoader(refiner_dataset, batch_size=32, shuffle=True)

# Fetch one batch
snippets, targets = next(iter(refiner_loader))

print(f"\nTotal Action Snippets Extracted: {len(refiner_dataset)}")
print(f"Batch Snippet Shape: {snippets.shape} -> (Batch, Sequence_Length, Features)")
print(f"Batch Targets Shape: {targets.shape} -> (Batch, [Start_Offset, End_Offset])")

# Look at the first snippet's target values
print(f"Sample Normalized Target: Start {targets[0][0]:.4f}, End {targets[0][1]:.4f}")

In [ ]:
import torch
import torch.nn as nn
import math

class TransformerRefiner(nn.Module):
    def __init__(self, input_dim=2048, d_model=256, nhead=4, num_layers=1, window_size=64):
        super(TransformerRefiner, self).__init__()
        
        # 1. Dimensionality Reduction
        self.input_proj = nn.Linear(input_dim, d_model)
        
        # 2. NEW: Local Temporal Inductive Bias
        # This helps the Transformer see local frame-to-frame motion changes
        self.local_conv = nn.Conv1d(
            in_channels=d_model, 
            out_channels=d_model, 
            kernel_size=3, 
            padding=1, 
            groups=d_model
        )
        self.norm = nn.LayerNorm(d_model)
        
        # 3. Positional Encoding
        self.pos_encoder = nn.Parameter(torch.zeros(1, window_size, d_model))
        
        # 4. The Dense Attention Block
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=d_model * 4, 
            dropout=0.1, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 5. The Boundary Regressor Head
        self.regressor = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Sigmoid() 
        )

    def forward(self, x):
        # x shape: (Batch, 64, 2048)
        x = self.input_proj(x) # Shape: (Batch, 64, 256)
        
        # Apply Local Convolution smoothing
        x_t = x.transpose(1, 2)
        x_t = self.local_conv(x_t)
        x_conv = x_t.transpose(1, 2)
        
        # Residual connection + Norm, then add Positional Encoding
        x = self.norm(x + x_conv) + self.pos_encoder
        
        # Apply dense self-attention
        x = self.transformer(x) # Shape: (Batch, 64, 256)
        
        # Temporal Pooling & Predict
        x_pooled = x.mean(dim=1) # Shape: (Batch, 256)
        boundaries = self.regressor(x_pooled) # Shape: (Batch, 2)
        
        return boundaries

In [ ]:
class GIoULoss1D(nn.Module):
    def __init__(self, eps=1e-6):
        super(GIoULoss1D, self).__init__()
        self.eps = eps

    def forward(self, preds, targets):
        # Ensure start is always strictly before end
        pred_start = torch.min(preds[:, 0], preds[:, 1])
        pred_end = torch.max(preds[:, 0], preds[:, 1])
        
        target_start = targets[:, 0]
        target_end = targets[:, 1]

        # Calculate Intersection
        intersect_start = torch.max(pred_start, target_start)
        intersect_end = torch.min(pred_end, target_end)
        intersect = torch.clamp(intersect_end - intersect_start, min=0)

        # Calculate Union
        pred_area = pred_end - pred_start
        target_area = target_end - target_start
        union = pred_area + target_area - intersect + self.eps

        # Calculate basic IoU
        iou = intersect / union

        # Calculate the smallest enclosing window
        enclose_start = torch.min(pred_start, target_start)
        enclose_end = torch.max(pred_end, target_end)
        enclose_area = torch.clamp(enclose_end - enclose_start, min=self.eps)

        # Calculate GIoU
        giou = iou - ((enclose_area - union) / enclose_area)

        # We want to minimize the loss, so 1 - GIoU
        loss = 1.0 - giou
        
        return loss.mean()

In [ ]:
import torch.optim as optim
from tqdm.auto import tqdm

# Hyperparameters
EPOCHS = 15
LEARNING_RATE = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Training Refiner on: {DEVICE}")

# Initialize the UPGRADED Model and Loss
refiner_model = TransformerRefiner().to(DEVICE)
criterion = GIoULoss1D().to(DEVICE) 
optimizer = optim.AdamW(refiner_model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

def train_refiner_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_mae = 0 

    pbar = tqdm(dataloader, desc="Training Refiner")

    for snippets, targets in pbar:
        snippets, targets = snippets.to(device), targets.to(device)

        # Forward Pass
        optimizer.zero_grad()
        predictions = model(snippets)
        
        # Calculate 1D GIoU Loss
        loss = criterion(predictions, targets)
        
        # Backward Pass & Optimize
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Track Mean Absolute Error (MAE) for human-readable progress
        with torch.no_grad():
            mae = torch.abs(predictions - targets).mean().item()
            total_mae += mae

        pbar.set_postfix(Loss=f"{loss.item():.4f}", MAE=f"{mae:.4f}")

    avg_loss = total_loss / len(dataloader)
    avg_mae = total_mae / len(dataloader)
    
    return avg_loss, avg_mae

# Run the Training Loop
for epoch in range(1, EPOCHS + 1):
    avg_loss, avg_mae = train_refiner_epoch(refiner_model, refiner_loader, optimizer, criterion, DEVICE)
    print(f"Epoch {epoch} Complete | Avg Loss: {avg_loss:.4f} | Avg MAE: {avg_mae:.4f}")

# Save the trained refiner!
torch.save(refiner_model.state_dict(), "transformer_refiner_stage2.pth")
print("\nRefiner Model saved successfully!")

In [ ]:
import os
import json
import numpy as np
import torch
from tqdm.auto import tqdm

# --- Configuration ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = '/kaggle/input/datasets/valuejack/thumos14-asl/THUMOS14' # Update if needed
SPLIT = 'test'
FPS = 25.0
CLIP_LENGTH = 16
WINDOW_SIZE = 64

print("Loading Models for Inference...")

# 1. Load the Trained Models
mamba_model = MambaGlobalScanner().to(DEVICE)
mamba_model.load_state_dict(torch.load("mamba_scanner_stage1.pth", weights_only=True))
mamba_model.eval()

refiner_model = TransformerRefiner().to(DEVICE)
refiner_model.load_state_dict(torch.load("transformer_refiner_stage2.pth", weights_only=True))
refiner_model.eval()

# 2. Setup Test Data Paths
rgb_dir = os.path.join(DATA_ROOT, 'features', SPLIT, 'rgb')
flow_dir = os.path.join(DATA_ROOT, 'features', SPLIT, 'flow')

with open(os.path.join(DATA_ROOT, f'split_{SPLIT}.txt'), 'r') as f:
    test_videos = [line.strip() for line in f.readlines()]

final_predictions = {}

# 3. The Inference Loop
with torch.no_grad():
    for vid_id in tqdm(test_videos, desc="Evaluating Test Set"):
        try:
            # Load Full Video Features
            rgb_feat = np.load(os.path.join(rgb_dir, f'{vid_id}.npy'))
            flow_feat = np.load(os.path.join(flow_dir, f'{vid_id}.npy'))
            video_features = torch.tensor(np.concatenate((rgb_feat, flow_feat), axis=-1), dtype=torch.float32).unsqueeze(0).to(DEVICE)
        except FileNotFoundError:
            continue
            
        num_vecs = video_features.shape[1]
        if num_vecs == 0: continue

        final_predictions[vid_id] = []

        # --- STAGE I: Global Scan ---
        mamba_logits = mamba_model(video_features)
        mamba_probs = torch.sigmoid(mamba_logits).squeeze(0).cpu().numpy()
        
        # Find candidate segments (probability > 0.5)
        # We look for continuous blocks of 1s in a boolean array
        is_action = mamba_probs > 0.5
        
        # A clever way to find start/end indices of contiguous blocks
        padded = np.pad(is_action, (1, 1), 'constant')
        diffs = np.diff(padded.astype(int))
        starts = np.where(diffs == 1)[0]
        ends = np.where(diffs == -1)[0] - 1 

        # --- STAGE II: Local Refinement ---
        for s_idx, e_idx in zip(starts, ends):
            # 1. Determine the 64-frame window center
            center_idx = (s_idx + e_idx) // 2
            w_start = center_idx - (WINDOW_SIZE // 2)
            w_end = w_start + WINDOW_SIZE
            
            # 2. Extract and Pad the Snippet
            snippet = torch.zeros((1, WINDOW_SIZE, 2048), dtype=torch.float32).to(DEVICE)
            
            safe_start = max(0, w_start)
            safe_end = min(num_vecs, w_end)
            
            insert_start = safe_start - w_start
            insert_end = insert_start + (safe_end - safe_start)
            
            if safe_end > safe_start:
                snippet[0, insert_start:insert_end] = video_features[0, safe_start:safe_end]
                
            # 3. Predict Normalized Boundaries
            pred_bounds = refiner_model(snippet).squeeze(0).cpu().numpy()
            t_start, t_end = pred_bounds[0], pred_bounds[1]
            
            # 4. Reverse Mapping: Normalized (0-1) -> Absolute Indices
            abs_start_idx = w_start + (t_start * WINDOW_SIZE)
            abs_end_idx = w_start + (t_end * WINDOW_SIZE)
            
            # Ensure boundaries are logical
            abs_start_idx = max(0, min(num_vecs, abs_start_idx))
            abs_end_idx = max(0, min(num_vecs, abs_end_idx))
            
            if abs_start_idx >= abs_end_idx:
                continue # Model predicted an invalid or collapsed window
                
            # 5. Final Conversion: Indices -> Absolute Seconds
            start_sec = (abs_start_idx * CLIP_LENGTH) / FPS
            end_sec = (abs_end_idx * CLIP_LENGTH) / FPS
            
            # Use Mamba's peak probability inside this window as the confidence score
            confidence = float(np.max(mamba_probs[s_idx:e_idx+1]))
            
            final_predictions[vid_id].append({
                "segment": [float(start_sec), float(end_sec)],
                "score": confidence
            })

# Save results to JSON for mAP evaluation
with open("submission_predictions.json", "w") as f:
    json.dump({"results": final_predictions}, f, indent=4)

print("\nInference Complete! Saved to submission_predictions.json")

In [ ]:
import json
import numpy as np

# --- Configuration ---
GT_FILE = '/kaggle/input/datasets/valuejack/thumos14-asl/THUMOS14/gt.json' # Update to your exact gt.json path
PRED_FILE = 'submission_predictions.json'
IOU_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7]

def compute_1d_iou(segment1, segment2):
    """Calculates Intersection over Union for two 1D temporal segments."""
    intersection_start = max(segment1[0], segment2[0])
    intersection_end = min(segment1[1], segment2[1])
    intersection = max(0, intersection_end - intersection_start)
    
    union = (segment1[1] - segment1[0]) + (segment2[1] - segment2[0]) - intersection
    if union <= 0: return 0.0
    return intersection / union

# 1. Load Data
with open(GT_FILE, 'r') as f:
    ground_truth = json.load(f)['database']
    
with open(PRED_FILE, 'r') as f:
    predictions = json.load(f)['results']

print("Calculating Average Precision (AP) at various IoU thresholds...\n")

# 2. Evaluation Loop
ap_scores = []

for iou_thresh in IOU_THRESHOLDS:
    all_scores = []
    all_matches = [] # 1 for True Positive, 0 for False Positive
    total_gt_actions = 0
    
    for vid_id, preds in predictions.items():
        if vid_id not in ground_truth: continue
            
        gt_annos = ground_truth[vid_id]['annotations']
        
        # THE FIX: Cast the JSON strings to floats
        gt_segments = [[float(ann['segment'][0]), float(ann['segment'][1])] for ann in gt_annos]
        total_gt_actions += len(gt_segments)
        
        # Sort predictions by confidence score (descending)
        preds = sorted(preds, key=lambda x: x['score'], reverse=True)
        
        gt_matched = [False] * len(gt_segments)
        
        for p in preds:
            all_scores.append(p['score'])
            
            best_iou = 0
            best_gt_idx = -1
            
            # Find the best matching ground truth segment
            for i, gt_seg in enumerate(gt_segments):
                if gt_matched[i]: continue
                iou = compute_1d_iou(p['segment'], gt_seg)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = i
                    
            if best_iou >= iou_thresh:
                all_matches.append(1)
                gt_matched[best_gt_idx] = True # Mark this GT as found
            else:
                all_matches.append(0)

    # 3. Calculate Precision-Recall Curve
    # Sort all predictions globally by confidence score
    sorted_indices = np.argsort(all_scores)[::-1]
    all_matches = np.array(all_matches)[sorted_indices]
    
    cumulative_tp = np.cumsum(all_matches)
    cumulative_fp = np.cumsum(1 - all_matches)
    
    precision = cumulative_tp / (cumulative_tp + cumulative_fp + 1e-8)
    recall = cumulative_tp / (total_gt_actions + 1e-8)
    
    # 4. Calculate Average Precision (Area under the PR curve)
    ap = 0.0
    # 11-point interpolation mapping
    for t in np.arange(0, 1.1, 0.1):
        if np.sum(recall >= t) == 0:
            p = 0
        else:
            p = np.max(precision[recall >= t])
        ap += p / 11.0
        
    ap_scores.append(ap)
    print(f"AP @ IoU {iou_thresh:.1f} : {ap * 100:.2f}%")

print("-" * 30)
print(f"Average mAP : {np.mean(ap_scores) * 100:.2f}%")

In [ ]:
# 1. Clone the gold-standard extraction repository
!git clone https://github.com/v-iashin/video_features.git
%cd video_features

# 2. Install the specific dependencies
!pip install omegaconf
!pip install timm
!pip install moviepy

In [ ]:
!find /kaggle/input -name "*istockphoto*.npy"

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os

# --- 1. MODEL ARCHITECTURES (Must be defined to load weights) ---

class MambaGlobalScanner(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=512):
        super().__init__()
        self.proj = nn.Linear(input_dim, hidden_dim)
        self.rnn = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim * 2, 1)
    def forward(self, x):
        x = self.proj(x)
        x, _ = self.rnn(x)
        return self.classifier(x)

class TransformerRefiner(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=512, nhead=8):
        super().__init__()
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=input_dim, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.encoder_layer, num_layers=1)
        self.fc = nn.Linear(input_dim, 2)
    def forward(self, x):
        x = self.transformer(x)
        x = torch.mean(x, dim=1)
        return torch.sigmoid(self.fc(x))

# --- 2. SETUP & LOADING ---

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FPS = 23.97
CLIP_LENGTH = 16 
WINDOW_SIZE = 64

# REPLACE THESE with the exact output from your !find command
rgb_path = "PASTE_YOUR_RGB_PATH_HERE.npy"
flow_path = "PASTE_YOUR_FLOW_PATH_HERE.npy"

print("Initializing Models and Loading Weights...")
mamba_model = MambaGlobalScanner().to(DEVICE)
mamba_model.load_state_dict(torch.load("mamba_scanner_stage1.pth", map_location=DEVICE))

refiner_model = TransformerRefiner().to(DEVICE)
refiner_model.load_state_dict(torch.load("transformer_refiner_stage2.pth", map_location=DEVICE))

mamba_model.eval()
refiner_model.eval()

# --- 3. FEATURE PROCESSING ---

print("Loading Extracted Features...")
rgb_feat = np.load(rgb_path)
flow_feat = np.load(flow_path)
fused_features = np.concatenate((rgb_feat, flow_feat), axis=-1)
video_tensor = torch.tensor(fused_features, dtype=torch.float32).unsqueeze(0).to(DEVICE)
num_vecs = video_tensor.shape[1]

# --- 4. INFERENCE ---

with torch.no_grad():
    mamba_logits = mamba_model(video_tensor)
    mamba_probs = torch.sigmoid(mamba_logits).squeeze(0).cpu().numpy()
    
    is_action = mamba_probs > 0.5
    padded = np.pad(is_action, (1, 1), 'constant')
    diffs = np.diff(padded.astype(int))
    starts = np.where(diffs == 1)[0]
    ends = np.where(diffs == -1)[0] - 1 

    final_predictions = []

    for s_idx, e_idx in zip(starts, ends):
        center_idx = (s_idx + e_idx) // 2
        w_start = center_idx - (WINDOW_SIZE // 2)
        w_end = w_start + WINDOW_SIZE
        
        snippet = torch.zeros((1, WINDOW_SIZE, 2048), dtype=torch.float32).to(DEVICE)
        safe_start, safe_end = max(0, w_start), min(num_vecs, w_end)
        insert_start = safe_start - w_start
        insert_end = insert_start + (safe_end - safe_start)
        
        if safe_end > safe_start:
            snippet[0, insert_start:insert_end] = video_tensor[0, safe_start:safe_end]
            
        pred_bounds = refiner_model(snippet).squeeze(0).cpu().numpy()
        abs_start_idx = max(0, min(num_vecs, w_start + (pred_bounds[0] * WINDOW_SIZE)))
        abs_end_idx = max(0, min(num_vecs, w_start + (pred_bounds[1] * WINDOW_SIZE)))
        
        if abs_start_idx < abs_end_idx:
            start_sec = (abs_start_idx * CLIP_LENGTH) / FPS
            end_sec = (abs_end_idx * CLIP_LENGTH) / FPS
            conf = float(np.max(mamba_probs[s_idx:e_idx+1]))
            final_predictions.append((start_sec, end_sec, conf))

print("\n" + "="*30)
print("FINAL DEMO PREDICTIONS")
print("="*30)
if not final_predictions:
    print("No action detected. (Try lowering the threshold from 0.5 if it's a subtle action)")
else:
    for i, (start, end, conf) in enumerate(final_predictions):
        print(f"Action {i+1}: {start:.2f}s to {end:.2f}s (Confidence: {conf*100:.1f}%)")